<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 2 · DATA WAREHOUSING WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">Observe Storage and Write Batches</div>
</div>

Write the ten orders of Module 1 into two tables with the same definition: once as one batch of ten rows and once as ten writes of one row. Both tables must give the same business result; then you read what the two methods left in storage, a tablet's versions and rowsets, and use `EXPLAIN` and the Query Profile to see what a query planned and what its scan actually read. Plan about 20 minutes.

**Module 2: Doris Storage Architecture and Write Mechanisms** · [Lesson](course.md) · [Quiz](quiz02_storage_and_write_batches.ipynb) · [Course README](../../README.md)

## Lab scope

The Lab rebuilds only `orders_batch` and `orders_rowwise` in your lab database. It needs the course sandbox from Lab 1 and no other table. Background compaction keeps reorganizing the rowsets, so some storage details depend on when you look; each Expected result says which.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import expect
from dw_course.wwi import HISTORY_COLUMNS as ORDER_COLUMNS, history_ddl as order_ddl, history_rows
from dw_course.ui import show_sql, show_records

lab = connect_sandbox()

## 1. Write the same ten orders in two ways

Both tables have the same columns, one bucket, and one replica; only the way of writing differs. `orders_batch` gets one `INSERT` with ten rows, and `orders_rowwise` ten `INSERT` statements with one row each. Each `INSERT` is one load transaction: once it commits, its rows are visible, and it adds one **version** to the tablet that holds them.

The connection set `group_commit = 'off_mode'` for this session, so each statement commits on its own. Module 5 shows Group Commit, which can merge many small writes like these into one transaction.

In [ ]:
lab.execute("DROP TABLE IF EXISTS orders_batch")
lab.execute("DROP TABLE IF EXISTS orders_rowwise")
ddl = order_ddl("orders_batch")
show_sql("Both tables: orders_rowwise differs only in its name", ddl)
lab.execute(ddl)
lab.execute(order_ddl("orders_rowwise"))

columns = ", ".join(ORDER_COLUMNS)
values = [f"({order_id}, {customer_id}, '{order_date}', {amount}, {lines}, '{source}')"
          for order_id, customer_id, order_date, amount, lines, source in history_rows()]
batch = f"INSERT INTO orders_batch ({columns}) VALUES\n" + ",\n".join(values)
show_sql("One write of ten rows", batch)
lab.execute(batch)
single_rows = [f"INSERT INTO orders_rowwise ({columns}) VALUES {value}" for value in values]
show_sql("Ten writes of one row: the first two", ";\n".join(single_rows[:2]) + ";\n-- ... and eight more, one per order")
for statement in single_rows:
    lab.execute(statement)

**Expected result**

The table definition, the batch `INSERT` with its ten rows, and the first two of the ten single-row statements. Nothing else: `lab.execute` reports nothing when a statement succeeds.

## 2. Check the business result first

Before comparing storage, check that both write methods saved the same orders: the same row count and pre-tax amount, 10 rows and 12220.60, as in Module 1. Different storage must never mean a different answer.

In [ ]:
for table in ("orders_batch", "orders_rowwise"):
    expect(lab.query(f"SELECT COUNT(*), SUM(order_amount) FROM {table}"), [(10, "12220.60")],
           title=f"{table}: ten orders, 12220.60")
lab.sql("""SELECT 'batch' AS write_mode, COUNT(*) AS orders, SUM(order_amount) AS amount FROM orders_batch
UNION ALL
SELECT 'rowwise', COUNT(*), SUM(order_amount) FROM orders_rowwise
ORDER BY write_mode""", title="Business results of both write methods");

**Expected result**

Two green checks and two rows, `batch` and `rowwise`, each with 10 orders and 12220.60.

## 3. Versions in the partition and the tablet

`SHOW PARTITIONS` and `SHOW TABLETS` read the FE's metadata. A table without `PARTITION BY` still has one partition, named after the table, and `BUCKETS 1` gives that partition one tablet. `VisibleVersion` of the partition and `Version` of the tablet count the committed loads: an empty tablet starts at version 1, and each load adds one.

In [ ]:
for table in ("orders_batch", "orders_rowwise"):
    lab.sql(f"SHOW PARTITIONS FROM {table}", title=f"The partition of {table}",
            columns=["PartitionName", "VisibleVersion", "Buckets", "ReplicationNum"])
    lab.sql(f"SHOW TABLETS FROM {table}", title=f"The tablet of {table}",
            columns=["TabletId", "Version", "VisibleVersionCount", "VersionCount", "RowCount"])

**Expected result**

Each table has one partition with one bucket and one replica, and one tablet. `orders_batch` is at version 2: version 1 plus one load. `orders_rowwise` is at version 11: version 1 plus ten loads. Ten writes of one row made ten transactions where one batch made one.

`VisibleVersionCount`, `VersionCount`, and `RowCount` come from the BE's tablet report, which arrives about once a minute: right after the writes they show `-1`, `-1`, and `0`, and a minute later the real numbers. `Version` is the FE's own count and is right immediately.

## 4. Rowsets, and what compaction does with them

Each load writes a **rowset** into the tablet: a versioned set of rows in one or more **segment** files. `SHOW TABLET <id>` finds where a tablet belongs and gives a `DetailCmd`, a `SHOW PROC` statement that lists its replicas with a `CompactionStatus` URL. That URL on the BE returns the tablet's rowsets. `lab.rowsets(table)` reads it for each tablet of a table and shows the request; the URL names the BE's address inside the container, `127.0.0.1:8040`, so it reads the same path from the host's BE port, 51040.

**Compaction** merges small rowsets in the background into larger ones; queries then read fewer files. It changes how the rows are stored, never which rows a query returns.

In [ ]:
tablet = lab.query("SHOW TABLETS FROM orders_rowwise")[0][0]
detail = lab.sql(f"SHOW TABLET {tablet}", title="Where the tablet of orders_rowwise belongs",
                 columns=["DbName", "TableName", "PartitionName", "IndexName", "DetailCmd"])
lab.sql(detail["DetailCmd"][0].rstrip(";"), title="Its replica, from the DetailCmd",
        columns=["ReplicaId", "BackendId", "Version", "State", "CompactionStatus"])
lab.rowsets("orders_batch")
lab.rowsets("orders_rowwise");

**Expected result**

The tablet belongs to `orders_rowwise`, and its replica on the one BE is at version 11, `NORMAL`, with the `CompactionStatus` URL.

`orders_batch` has two current rowsets: `[0-1]` with 0 segments, the empty start of every tablet, and `[2-2]` with 1 segment and about 1.5 KB, the ten rows of the batch.

`orders_rowwise` shows one of three moments. Right after the writes, ten current rowsets `[2-2]` to `[11-11]` of about 1.5 KB each. Usually, after a second or two, cumulative compaction has merged them: one current rowset `[2-11]` and the ten old ones listed as stale, which the BE deletes some minutes later. After that, only `[0-1]` and `[2-11]`. `[2-11]` is a range of versions, not ten orders. In all three moments the query in step 2 returns the same ten orders.

## 5. The plan and the work: EXPLAIN and the Query Profile

`EXPLAIN` shows the plan that the FE built for a query, without running it: the table to scan, the columns, and the filter. The **Query Profile** records what each operator actually did when the query ran: how many rows the scan read, how many it skipped and how, and how many bytes it read.

The next cell turns on Profiles for this session and turns off the SQL cache, so that a repeated query really scans, and restores both settings afterwards. `lab.scan_profile(query)` runs a query, takes its query ID with `SELECT last_query_id()` (a query's Profile ID is its query ID), fetches the Profile's text from the FE, and adds up the scan's row counters. It compares three queries on `orders_batch`, whose sort key is `order_id`:

| Counter | Meaning |
| --- | --- |
| `ScanRows` | Rows that the scan read from its segments |
| `RowsKeyRangeFiltered` | Rows that the sort key's index let the scan skip without reading |
| `RowsStatsFiltered` | Rows skipped with the zone maps, the minimum and maximum of each page and segment |
| `RowsVectorPredFiltered` | Rows that were read and then dropped by the filter |
| `RowsRead` | Rows that passed the filter: `ScanRows` minus `RowsVectorPredFiltered` |

The FE's Web UI shows the same Profiles: open `http://127.0.0.1:51030` (or the port in `DW_FE_HTTP_PORT`), sign in as `root` with an empty password, and choose **QueryProfile**.

In [ ]:
lab.sql("EXPLAIN SELECT order_id, order_amount FROM orders_batch WHERE order_id = 1",
        title="The plan: the table, the columns, and the filter");
previous = lab.query("SELECT @@enable_profile, @@enable_sql_cache")[0]
lab.execute("SET enable_profile = true, enable_sql_cache = false")
try:
    scans = [lab.scan_profile(statement) for statement in (
        "SELECT order_id, order_amount FROM orders_batch WHERE order_id = 1",
        "SELECT order_id, order_amount FROM orders_batch WHERE customer_id = 832",
        "SELECT * FROM orders_batch WHERE customer_id = 832",
    )]
finally:
    lab.execute("SET enable_profile = %s, enable_sql_cache = %s", previous)
show_records("What the scan of each query read", scans)

**Expected result**

The plan's scan node names `orders_batch`, the predicate `order_id = 1`, and `partitions=1/1`, `tablets=1/1`: a plan, not a count of what was read. Then the three statements with their query IDs and the curl requests for their Profiles, and a table with one row per query:

| Query | rows_returned | ScanRows | RowsKeyRangeFiltered | RowsVectorPredFiltered |
| --- | ---: | ---: | ---: | ---: |
| `WHERE order_id = 1`, two columns | 1 | 1 | 9 | 0 |
| `WHERE customer_id = 832`, two columns | 1 | 10 | 0 | 9 |
| `WHERE customer_id = 832`, all columns | 1 | 10 | 0 | 9 |

All three return order 1, but they do different work. The filter on the sort key's column lets the index narrow the scan to one row. The filter on `customer_id`, not part of the key, makes the scan read all ten rows and drop nine. With the same filter, the third query reads all six columns instead of two, and its `ScanBytes` is clearly larger, a few hundred bytes more: a column store reads only the columns that a query uses. On ten rows these are tiny amounts and the times say nothing about speed; the counters show what kind of work a query does, and the same counters grow with the data.

## 6. Practice: one date in both tables

Query the orders of 2013-01-01 in both tables and show them in one result; expect 5 orders and 3944.20 for each table. Then read the scan counters of `SELECT COUNT(*), SUM(order_amount) FROM orders_batch WHERE order_date = '2013-01-01'`, with Profiles on as in step 5, and explain the numbers: does the sort key help this filter?

In [ ]:
# Your code here

The next cell is a reference solution; it is collapsed. Expand it to compare, or run it.

In [ ]:
query = """SELECT 'batch' AS write_mode, COUNT(*) AS orders, SUM(order_amount) AS amount
FROM orders_batch WHERE order_date = '2013-01-01'
UNION ALL
SELECT 'rowwise', COUNT(*), SUM(order_amount)
FROM orders_rowwise WHERE order_date = '2013-01-01'
ORDER BY write_mode"""
lab.sql(query, title="First-day orders from both write methods");
expect(lab.query(query), [("batch", 5, "3944.20"), ("rowwise", 5, "3944.20")],
       title="Both tables: five orders, 3944.20")
previous = lab.query("SELECT @@enable_profile, @@enable_sql_cache")[0]
lab.execute("SET enable_profile = true, enable_sql_cache = false")
try:
    show_records("What the scan of the date filter read", [lab.scan_profile(
        "SELECT COUNT(*), SUM(order_amount) FROM orders_batch WHERE order_date = '2013-01-01'")])
finally:
    lab.execute("SET enable_profile = %s, enable_sql_cache = %s", previous)

**Expected result**

`batch` and `rowwise`, each with 5 orders and 3944.20. The scan of the date filter: `ScanRows` 10, `RowsRead` 5, `RowsVectorPredFiltered` 5, and nothing skipped by the key range or the zone maps. `order_date` is not part of the sort key `order_id`, so the index cannot narrow the scan, and both dates are in the same segment, so its minimum and maximum dates cannot rule it out either. Module 3 partitions orders by date, so that a query for one day skips the other day's data.

## You are done when

- both tables hold 10 orders and 12220.60, and you can say why their versions differ: 2 for one batch, 11 for ten writes;
- you have read the rowsets of both tables and can explain what `[2-11]` means and what compaction changed;
- you can tell from a plan what a query intends to read, and from its Profile what its scan read and skipped.

The ten rows are a teaching sample: they show the mechanisms, not production speed. Measuring write throughput needs large data, fixed concurrency, and observation over time.

## Next

Open [Quiz 2: Doris Storage Architecture and Write Mechanisms](quiz02_storage_and_write_batches.ipynb), then continue to [Module 3: Doris Table Models, Partitioning, and Bucketing](../module03-table-design/course.md).